In [13]:
# ============================================
# CELL 1 — Install Libraries
# ============================================
!pip install rank_bm25 openai -q
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print("✅ Libraries installed!")

✅ Libraries installed!


In [2]:
# ================================
# CELL 2 — Load Data
# ================================
import pickle
from google.colab import files

print("Upload processed_data.pkl")
files.upload()

with open('processed_data.pkl', 'rb') as f:
    data = pickle.load(f)

statute_texts   = data['statute_texts']
statute_ids     = data['statute_ids']
precedent_texts = data['precedent_texts']
precedent_ids   = data['precedent_ids']

print("✅ Data loaded!")
print(f"Statutes  : {len(statute_texts)}")
print(f"Precedents: {len(precedent_texts)}")

Upload processed_data.pkl


Saving processed_data.pkl to processed_data.pkl
✅ Data loaded!
Statutes  : 936
Precedents: 3183


In [3]:
# ================================
# CELL 3 — Prepare Clean Texts
# ================================
def prepare_text(text, max_words=400):
    if isinstance(text, list):
        text = ' '.join(text)
    words = text.split()[:max_words]
    return ' '.join(words)

statute_texts_clean   = [prepare_text(t)
    for t in statute_texts]
precedent_texts_clean = [prepare_text(t)
    for t in precedent_texts]

print("✅ Clean texts ready!")

✅ Clean texts ready!


In [4]:
# ================================
# CELL 4 — Build BM25 Indexes
# ================================
from rank_bm25 import BM25Okapi
from nltk.tokenize import word_tokenize

def build_ngrams(tokens, n):
    if n == 1:
        return tokens
    return [' '.join(tokens[i:i+n])
            for i in range(len(tokens)-n+1)]

def tokenize_ngram(text, n):
    if isinstance(text, list):
        text = ' '.join(text)
    tokens = word_tokenize(text.lower())
    return build_ngrams(tokens, n)

print("Building statute index (3-gram)...")
bm25_statute_index = BM25Okapi(
    [tokenize_ngram(t, 3)
     for t in statute_texts],
    k1=1.6, b=0.75)
print("✅ Statute index ready!")

print("Building precedent index (5-gram)...")
bm25_precedent_index = BM25Okapi(
    [tokenize_ngram(t, 5)
     for t in precedent_texts],
    k1=1.6, b=0.75)
print("✅ Precedent index ready!")

Building statute index (3-gram)...
✅ Statute index ready!
Building precedent index (5-gram)...
✅ Precedent index ready!


In [28]:
# ============================================
# CELL 5 — Setup Groq (xAI)
# ============================================
from openai import OpenAI
from google.colab import userdata

# Initialize Groq client
client = OpenAI(
    api_key=userdata.get('GROQ_API_KEY'),
    base_url="https://api.groq.com/openai/v1",
)

# Test connection
try:
    test = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "user",
             "content": "Say hello in one word"}
        ]
    )
    print("✅ Groq connected!")
    print(f"Test response: "
          f"{test.choices[0].message.content}")
except Exception as e:
    print(f"❌ Error: {e}")
    print("Check your GROQ_API_KEY in secrets! Also ensure the model 'llama-3.3-70b-versatile' is active.")

✅ Groq connected!
Test response: Hello


In [31]:
# ============================================
# CELL 6 — Retrieval Function
# ============================================
def retrieve_top_k(query, k=3):
    """
    Retrieve top-k statutes and precedents
    using pre-built BM25 indexes
    Very fast — 2-3 seconds per query
    """
    # Retrieve statutes (3-gram)
    query_tokens_3 = tokenize_ngram(query, 3)
    stat_scores = bm25_statute_index\
        .get_scores(query_tokens_3)
    top_stat_idx = sorted(
        range(len(stat_scores)),
        key=lambda x: stat_scores[x],
        reverse=True)[:k]

    top_statutes    = [statute_texts_clean[i]
                       for i in top_stat_idx]
    top_statute_ids = [statute_ids[i]
                       for i in top_stat_idx]

    # Retrieve precedents (5-gram)
    query_tokens_5 = tokenize_ngram(query, 5)
    prec_scores = bm25_precedent_index\
        .get_scores(query_tokens_5)
    top_prec_idx = sorted(
        range(len(prec_scores)),
        key=lambda x: prec_scores[x],
        reverse=True)[:k]

    top_precedents    = [precedent_texts_clean[i]
                         for i in top_prec_idx]
    top_precedent_ids = [precedent_ids[i]
                         for i in top_prec_idx]

    return (top_statutes, top_statute_ids,
            top_precedents, top_precedent_ids)

print("✅ Retrieval function ready!")

✅ Retrieval function ready!


In [32]:
# ============================================
# CELL 7 — RAG Pipeline Function
# ============================================
def rag_pipeline(query):
    """
    Full RAG Pipeline:
    1. Retrieve top-3 statutes + precedents
    2. Build context from retrieved docs
    3. Send to Grok for plain English explanation
    4. Return citizen-friendly legal explanation

    Original contribution — not in IL-PCSR paper!
    """
    print(f"\n{'='*60}")
    print(f"QUERY: {query[:200]}")
    print(f"{'='*60}")

    # Step 1 — Retrieve
    print("\n📚 Retrieving relevant documents...")
    (top_statutes, statute_ids_ret,
     top_precedents, precedent_ids_ret) = \
        retrieve_top_k(query, k=3)

    print(f"✅ Found {len(top_statutes)} "
          f"statutes and "
          f"{len(top_precedents)} precedents")

    # Step 2 — Build context
    context  = "RELEVANT STATUTES:\n"
    context += "="*40 + "\n"
    for i, stat in enumerate(top_statutes):
        context += f"\nStatute {i+1} "
        context += f"(ID: {statute_ids_ret[i]}):\n"
        context += stat[:600] + "\n"
        context += "-"*40 + "\n"

    context += "\n\nRELEVANT PRECEDENTS:\n"
    context += "="*40 + "\n"
    for i, prec in enumerate(top_precedents):
        context += f"\nPrecedent {i+1} "
        context += f"(ID: {precedent_ids_ret[i]}):\n"
        context += prec[:600] + "\n"
        context += "-"*40 + "\n"

    # Step 3 — Build prompt
    prompt = f"""You are a helpful legal assistant
helping common citizens understand their legal rights.

A person has described their legal situation below.
Based ONLY on the relevant statutes and precedents
provided, explain clearly:

1. Which specific laws apply to their situation
2. What exactly those laws say in simple words
3. What similar court cases have decided
4. What practical steps they can take next

IMPORTANT RULES:
- Use very simple language — explain like to a
  person with no legal knowledge
- Be specific about which law/section applies
- Be practical and actionable
- Keep explanation under 300 words
- End with this exact disclaimer:
  "⚠️ DISCLAIMER: This explanation is for
  informational purposes only and does not
  constitute legal advice. Please consult a
  qualified lawyer for your specific situation."

PERSON'S LEGAL SITUATION:
{query}

{context}

Now provide your clear, helpful explanation:"""

    # Step 4 — Generate with Grok
    try:
        print("\n🤖 Generating explanation with Grok...")
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {
                    "role"   : "system",
                    "content": "You are a helpful "
                               "legal assistant that "
                               "explains laws in "
                               "simple plain English "
                               "for common citizens."
                },
                {
                    "role"   : "user",
                    "content": prompt
                }
            ],
            max_tokens=500,
            temperature=0.3
        )

        result = response.choices[0]\
            .message.content

        print("\n" + "="*60)
        print("📋 LEGAL EXPLANATION:")
        print("="*60)
        print(result)
        print("="*60)

        return {
            'query'           : query,
            'explanation'     : result,
            'statute_ids'     : statute_ids_ret,
            'precedent_ids'   : precedent_ids_ret,
            'top_statutes'    : top_statutes,
            'top_precedents'  : top_precedents
        }

    except Exception as e:
        print(f"❌ Error: {e}")
        return None

print("✅ RAG pipeline ready!")

✅ RAG pipeline ready!


In [33]:
# ============================================
# CELL 8 — Test RAG on 3 Queries
# ============================================
import time

rag_results = []

# ── Test 1 — Employment Issue ─────────────
print("\n" + "🧪"*30)
print("TEST 1 — Employment Issue")
print("🧪"*30)

result1 = rag_pipeline("""
My employer has not paid my salary for
the last 3 months. I have asked him
multiple times but he keeps making excuses.
I have worked for this company for 2 years.
What law protects me and what can I do?
""")
if result1:
    rag_results.append(result1)

print("\nWaiting 10 seconds...")
time.sleep(10)

# ── Test 2 — Tenant Issue ─────────────────
print("\n" + "🧪"*30)
print("TEST 2 — Tenant Issue")
print("🧪"*30)

result2 = rag_pipeline("""
My landlord illegally cut my electricity
without any notice or court order.
I have been living here for 5 years and
always paid rent on time.
What are my legal rights?
""")
if result2:
    rag_results.append(result2)

print("\nWaiting 10 seconds...")
time.sleep(10)

# ── Test 3 — Consumer Fraud ───────────────
print("\n" + "🧪"*30)
print("TEST 3 — Consumer Fraud")
print("🧪"*30)

result3 = rag_pipeline("""
I bought a mobile phone online for
Rs. 50,000 and the seller sent me a
fake/counterfeit item. He is refusing
to refund my money and has blocked me.
What can I do legally?
""")
if result3:
    rag_results.append(result3)

print(f"\n✅ ALL RAG TESTS COMPLETE!")
print(f"Successfully processed: "
      f"{len(rag_results)}/3 queries")


🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪
TEST 1 — Employment Issue
🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪

QUERY: 
My employer has not paid my salary for 
the last 3 months. I have asked him 
multiple times but he keeps making excuses.
I have worked for this company for 2 years.
What law protects me and what can 

📚 Retrieving relevant documents...
✅ Found 3 statutes and 3 precedents

🤖 Generating explanation with Grok...

📋 LEGAL EXPLANATION:
The laws that apply to your situation are related to employment and payment of salaries. Specifically, Statute 3 (ID: 938899) defines what constitutes a salary, which includes wages and any other payments received by an employee. 

In simple words, this law says that your employer is required to pay you for the work you have done. Since you have not received your salary for the last 3 months, your employer is not following the law.

There aren't any directly relevant court cases provided, but you can still take action. You can start by sending a formal letter to 

In [34]:
# ============================================
# CELL 9 — Save RAG Results
# ============================================
import pickle
import json
from google.colab import files

# Save as pickle
with open('rag_results.pkl', 'wb') as f:
    pickle.dump(rag_results, f)

# Save as readable JSON
rag_json = []
for r in rag_results:
    if r:
        rag_json.append({
            'query'         : r['query'],
            'explanation'   : r['explanation'],
            'statute_ids'   : r['statute_ids'],
            'precedent_ids' : r['precedent_ids']
        })

with open('rag_results.json', 'w') as f:
    json.dump(rag_json, f,
              indent=2, ensure_ascii=False)

print("✅ Results saved!")

# Download both files
files.download('rag_results.pkl')
files.download('rag_results.json')
print("✅ Downloaded!")
print("\nrag_results.json is human readable")
print("Open it in notepad to see results!")

✅ Results saved!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloaded!

rag_results.json is human readable
Open it in notepad to see results!


In [35]:
# ============================================
# CELL 10 — Print Summary
# ============================================
print("\n" + "="*60)
print("RAG PIPELINE SUMMARY")
print("="*60)
print(f"\nModel used    : Groq (llama-3.3-70b-versatile)")
print(f"Retrieval     : BM25 (3-gram for "
      f"statutes, 5-gram for precedents)")
print(f"Statutes pool : 936 documents")
print(f"Precedent pool: 3,183 documents")
print(f"Top-k docs    : 3 statutes + "
      f"3 precedents per query")
print(f"\nQueries tested: {len(rag_results)}")

for i, r in enumerate(rag_results):
    if r:
        print(f"\nQuery {i+1}:")
        print(f"  Statutes retrieved  : "
              f"{r['statute_ids']}")
        print(f"  Precedents retrieved: "
              f"{r['precedent_ids']}")
        print(f"  Explanation length  : "
              f"{len(r['explanation'])} chars")

print("\n" + "="*60)
print("✅ RAG NOTEBOOK COMPLETE!")
print("="*60)
print("\nThis is your ORIGINAL contribution!")
print("IL-PCSR paper did NOT implement RAG!")
print("You extended their work! 🎉")


RAG PIPELINE SUMMARY

Model used    : Groq (llama-3.3-70b-versatile)
Retrieval     : BM25 (3-gram for statutes, 5-gram for precedents)
Statutes pool : 936 documents
Precedent pool: 3,183 documents
Top-k docs    : 3 statutes + 3 precedents per query

Queries tested: 3

Query 1:
  Statutes retrieved  : ['885027', '1591527', '938899']
  Precedents retrieved: ['1864509', '309285', '581019']
  Explanation length  : 1129 chars

Query 2:
  Statutes retrieved  : ['439618', '1183163', '1591527']
  Precedents retrieved: ['1864509', '309285', '581019']
  Explanation length  : 1379 chars

Query 3:
  Statutes retrieved  : ['1903729', '1454268', '1538044']
  Precedents retrieved: ['1864509', '309285', '581019']
  Explanation length  : 1144 chars

✅ RAG NOTEBOOK COMPLETE!

This is your ORIGINAL contribution!
IL-PCSR paper did NOT implement RAG!
You extended their work! 🎉
